# ELENDRIL V8-X2 — Palantiri 2xT4 Coordinated Research Campaign

Zero-edit owner-armed Kaggle 2xT4 notebook pinned to immutable commits:
- **X2 Runtime (`SOURCE_SHA`):** `596c5e3dd07cb30bd636b68250d11d960adf67ae` (`PLANTHIR_BACKEND_MODULES=experiments.elendril_x2`)
- **Legacy X1 Compatibility Lane (`LEGACY_X1_SHA`):** `8e62ca0f2046c0f07910fdd20abee40508631227`
- Select **Accelerator: GPU T4 x2**, enable **Internet**, and click **Run All** (no code edits required).
- **Combined 10-hour 2xT4 session (`600m` session / `660m` outer / `45m` reserve):** Run 1 covers Sessions 1 + 2 combined (up to 43 jobs across `A1..U1`); Run 2+ attaches Run 1's output ZIP to auto-resume the rest.
- **Run 1 requires NO `/kaggle/input` attachment** (X1 A0 evidence is already committed in `Experiment Result/V8-X1/A0/`).
- Enforces mandatory 2xT4 canary + independent `replay-bundle` verification gate before launching waves, stops cleanly at `9.25h` or on all-failing waves, prunes completed heavy `.bin` / optimizer / `checkpoint_full.zip` weight blobs after each wave, and refreshes a **single complete result archive** (`ELENDRIL_V8_X2_ANALYSIS.zip`) containing 100% of episodes, mistakes, trajectories, summaries, probes, Tier-2 `.npz` capsules, specs, and `result.zip`.


In [ ]:
# Owner-armed zero-edit defaults: select 2xT4 GPU and click Run All.
OWNER_AUTHORIZED = True
SESSIONS = ["A1", "A2", "A3", "D0", "D1", "D2", "U0", "U1", "IIIb-DM", "IIIb-P2", "IIIb-R2", "IIIb-U-P", "IIIb-PS", "BIV-T13", "BIV-T14", "BV-H4736"]
OUTER_MAX_MINUTES = 660
SESSION_MAX_MINUTES = 600
EXPORT_RESERVE_MINUTES = 45
MEASURED_SEC_PER_UPDATE = 0.65
PER_JOB_DISK_MB = 32
PRUNE_COMPLETED_WEIGHTS = True
RESULTS_ONLY_EXPORT = True
DEVICES = "0,1"
AUTO_RESUME_FROM_KAGGLE_INPUT = True
SOURCE_SHA = "596c5e3dd07cb30bd636b68250d11d960adf67ae"
LEGACY_X1_SHA = "8e62ca0f2046c0f07910fdd20abee40508631227"
REPO_URL = "https://github.com/dhurv0045com-spec/An-Ra-the-new-AGI.git"


In [ ]:
import os, shutil, subprocess, sys
from pathlib import Path

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["MKL_NUM_THREADS"] = "2"
os.environ["PLANTHIR_BACKEND_MODULES"] = "experiments.elendril_x2"

WORK_ROOT = Path("/tmp/elendril_x2")
REPO_X2_DIR = WORK_ROOT / "repo_x2"
REPO_X1_DIR = WORK_ROOT / "repo_x1_legacy"
OUTPUT_DIR = Path("/kaggle/working")

def _checkout_pinned(dest: Path, sha: str) -> None:
    if dest.exists():
        shutil.rmtree(dest)
    dest.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "-C", str(dest), "init", "-q"], check=True)
    subprocess.run(["git", "-C", str(dest), "remote", "add", "origin", REPO_URL], check=True)
    try:
        subprocess.run(["git", "-C", str(dest), "fetch", "--depth=1", "origin", sha], check=True)
    except subprocess.CalledProcessError:
        subprocess.run(["git", "-C", str(dest), "fetch", "origin", sha], check=True)
    subprocess.run(["git", "-C", str(dest), "checkout", "--detach", sha], check=True)
    head = subprocess.check_output(["git", "-C", str(dest), "rev-parse", "HEAD"], text=True).strip()
    assert head == sha, f"Checkout mismatch: expected {sha}, got {head}"

if not OWNER_AUTHORIZED:
    print("NOT LAUNCHED: set OWNER_AUTHORIZED = True in the configuration cell to execute X2.")
else:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    _checkout_pinned(REPO_X2_DIR, SOURCE_SHA)
    _checkout_pinned(REPO_X1_DIR, LEGACY_X1_SHA)
    os.chdir(REPO_X2_DIR)
    if str(REPO_X2_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_X2_DIR))
    prev_pp = os.environ.get("PYTHONPATH", "")
    parts = [p for p in prev_pp.split(os.pathsep) if p] if prev_pp else []
    if str(REPO_X2_DIR) not in parts:
        os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO_X2_DIR), *parts]) if parts else str(REPO_X2_DIR)
    print("Checked out X2 runtime:", SOURCE_SHA)
    print("Checked out legacy X1 worker:", LEGACY_X1_SHA)


In [ ]:
import json, shutil, subprocess, sys, torch
from pathlib import Path

if not OWNER_AUTHORIZED:
    print("NOT LAUNCHED: owner switch is false")
else:
    gpu_count = torch.cuda.device_count()
    print("Torch:", torch.__version__, "CUDA GPUs:", gpu_count)
    for idx in range(gpu_count):
        print(f"  cuda:{idx} -> {torch.cuda.get_device_name(idx)}")
    assert torch.cuda.is_available() and gpu_count >= 2, f"2xT4 CUDA GPUs required, found {gpu_count}"

    # Mandatory pre-flight gate: Palantiri doctor + 2-device X2 canary + independent replay-bundle
    subprocess.run([sys.executable, "-m", "anra_v5.palantiri", "doctor"], check=True)
    canary_dir = WORK_ROOT / "preflight_canary"
    canary_dir.mkdir(parents=True, exist_ok=True)
    canary_spec = canary_dir / "x2_canary_spec.json"
    subprocess.run([sys.executable, "-m", "anra_v5.palantiri", "init", "--backend", "elendril_x2", "--out", str(canary_spec)], check=True)
    doc = json.loads(canary_spec.read_text(encoding="utf-8"))
    doc["source_commit"] = SOURCE_SHA
    doc["execution"]["device"] = "cuda:0"
    canary_spec.write_text(json.dumps(doc, indent=2) + "\n", encoding="utf-8")
    subprocess.run([sys.executable, "-m", "anra_v5.palantiri", "validate", "--spec", str(canary_spec), "--canary"], check=True)
    subprocess.run([sys.executable, "-m", "anra_v5.palantiri", "run", "--spec", str(canary_spec), "--work-root", str(canary_dir / "run"), "--canary", "--device", "cuda:0"], check=True)
    subprocess.run([sys.executable, "-m", "anra_v5.palantiri", "verify-bundle", "--bundle-dir", str(canary_dir / "run" / "result_bundle"), "--expected-source-commit", SOURCE_SHA], check=True)
    subprocess.run([sys.executable, "-m", "anra_v5.palantiri", "replay-bundle", "--bundle-dir", str(canary_dir / "run" / "result_bundle"), "--expected-source-commit", SOURCE_SHA, "--device", "cuda:0", "--output", str(canary_dir / "REPLAY_RECEIPT.json")], check=True)
    camp_work = WORK_ROOT / "campaign_work"
    camp_work.mkdir(parents=True, exist_ok=True)
    shutil.copy2(canary_dir / "REPLAY_RECEIPT.json", camp_work / "PREFLIGHT_REPLAY_RECEIPT.json")
    shutil.rmtree(canary_dir / "run", ignore_errors=True)
    print("PREFLIGHT CANARY + INDEPENDENT REPLAY PASSED:", camp_work / "PREFLIGHT_REPLAY_RECEIPT.json")


In [ ]:
import json, shutil, time
from pathlib import Path
from experiments.x2.campaign import (
    ElendrilX2Campaign,
    build_verified_resume_set,
    discover_kaggle_input_receipts,
    run_a0_historical_rescore,
)

if not OWNER_AUTHORIZED:
    print("NOT LAUNCHED: owner switch is false")
else:
    t_start = time.monotonic()
    camp_work = WORK_ROOT / "campaign_work"
    camp = ElendrilX2Campaign(
        source_commit=SOURCE_SHA,
        work_dir=camp_work,
        legacy_x1_repo_dir=REPO_X1_DIR,
        outer_max_minutes=OUTER_MAX_MINUTES,
        session_max_minutes=SESSION_MAX_MINUTES,
        export_reserve_minutes=EXPORT_RESERVE_MINUTES,
    )
    camp.prepare(SESSIONS)

    # Auto-discover mounted /kaggle/input prior receipts (only needed for Session 2+ or A0 archive re-check)
    imported_receipts = []
    k_in = Path("/kaggle/input")
    if AUTO_RESUME_FROM_KAGGLE_INPUT and k_in.is_dir():
        for a0_zip in sorted(k_in.rglob("*A0*ANALYSIS*.zip")):
            print("Cross-checking mounted historical A0 ZIP:", a0_zip)
            run_a0_historical_rescore(a0_zip, camp_work / "a0_rescore.json")
        imported_receipts = discover_kaggle_input_receipts(k_in, camp_work / "imported_inputs", camp.registry)
        print("Discovered mounted resume receipts:", len(imported_receipts))
    resume_rec = build_verified_resume_set(registry=camp.registry, work_dir=camp_work, imported_receipts=imported_receipts)
    print("Verified completed jobs imported:", resume_rec["verified_count"])

    # Admit whole jobs fitting the governing dual-clock & disk budget and run on 2xT4
    plan = camp.admit_slice(
        SESSIONS,
        measured_seconds_per_update=MEASURED_SEC_PER_UPDATE,
        measured_per_job_disk_bytes=PER_JOB_DISK_MB * 1024 * 1024,
        parallel_workers=2,
        completed_ids=resume_rec["verified_completed_ids"],
    )
    print("Admitted jobs for this session:", len(plan["admitted_job_ids"]), "->", plan["admitted_job_ids"])
    admitted_set = set(plan["admitted_job_ids"])
    rows = [r for r in camp.select_session_rows(SESSIONS) if r["job_id"] in admitted_set]
    dev_list = [f"cuda:{d.strip()}" for d in DEVICES.split(",") if d.strip()]
    deadline_epoch = time.time() + camp.budget["usable_train_minutes"] * 60.0
    def _on_wave_complete(wave_idx: int, wave_rec: dict) -> None:
        if wave_rec.get("succeeded_count", 0) > 0:
            camp.export_archives(OUTPUT_DIR, prefix='ELENDRIL_V8_X2', status='PARTIAL', results_only=RESULTS_ONLY_EXPORT)
    summary = camp.run_admitted_waves(
        camp.to_session_jobs(rows),
        devices=dev_list,
        resume=bool(resume_rec["verified_completed_ids"]),
        canary=False,
        deadline_epoch=deadline_epoch,
        fail_fast_on_empty_wave=True,
        prune_completed_heavy_artifacts=PRUNE_COMPLETED_WEIGHTS,
        on_wave_complete=_on_wave_complete,
    )
    print(
        "CAMPAIGN STATUS:",
        json.dumps({
            "completed_count": summary["completed_count"],
            "failed_job_ids": summary["failed_job_ids"],
            "unfinished_job_ids": summary["unfinished_job_ids"],
            "stopped_reason": summary.get("stopped_reason"),
            "next_resumable_job_id": summary.get("next_resumable_job_id") or plan.get("next_resumable_job_id"),
        }, indent=2, sort_keys=True),
    )
    for diag in summary.get("failed_worker_diagnostics", ()):
        print(f"FAILED WORKER {diag['worker_id']} exit={diag['exit_code']} stderr_tail:\n{diag['stderr_tail']}")
    final_status = "COMPLETE" if (not summary["unfinished_job_ids"] and not plan["deferred_pending_job_ids"]) else "PARTIAL"
    delivery = camp.export_archives(OUTPUT_DIR, prefix='ELENDRIL_V8_X2', status=final_status, results_only=RESULTS_ONLY_EXPORT)
    print("DELIVERY RECEIPT:", json.dumps(delivery["final_status"], indent=2, sort_keys=True))
    print("NEXT RESUMABLE JOB ID:", summary.get("next_resumable_job_id") or plan.get("next_resumable_job_id"))
    if len(summary["completed_jobs"]) == 0 and summary["failed_job_ids"]:
        raise RuntimeError(f"Campaign had failed jobs with 0 completions: {summary['failed_job_ids']}")


In [ ]:
from IPython.display import FileLink, display
from pathlib import Path

if not OWNER_AUTHORIZED:
    print("NOT LAUNCHED: owner switch is false")
else:
    out = Path("/kaggle/working")
    for p in sorted(out.glob("ELENDRIL_V8_X2*.zip")) + sorted(out.glob("*.sha256")) + [out / "FINAL_STATUS.json", out / "DELIVERY_RECEIPT.json"]:
        if p.is_file():
            print(f"{p.name} ({p.stat().st_size:,} bytes)")
            display(FileLink(str(p)))
